# CLEAR Corpus Readability Dataset - Demo Notebook

This notebook demonstrates the CLEAR (CommonLit Ease of Readability) Corpus dataset for readability research.

## What is the CLEAR Corpus?

The CLEAR Corpus contains **4,724 text excerpts** with real human readability judgments from teachers, transformed to a 1-100 scale. Key features:

- **REAL human judgments** (not algorithmic)
- **N=4,724** > 1,000 examples
- **Diverse sources** spanning 250+ years
- **Multiple raters** per text via Rasch model
- **Standardized 1-100 scale**
- **Varied text lengths** (avg 172 words)
- **Permissive license** (CC BY-NC-SA 4.0)

## What This Notebook Does

1. Loads a mini subset of the CLEAR Corpus (3 examples)
2. Explores the data structure and metadata
3. Visualizes readability score distributions
4. Demonstrates basic dataset statistics

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print('Dependencies installed successfully!')

In [ ]:
# Imports
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Any

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6d018e-readability-as-circuit-resistance-a-nove/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    # Try loading from GitHub URL first
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            print("Loaded data from GitHub URL")
            return data
    except Exception as e:
        print(f"GitHub URL load failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            data = json.load(f)
            print("Loaded data from local file")
            return data
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined!")

In [ ]:
# Load the data
data = load_data()

# Extract examples from the datasets format
dataset_name = data['datasets'][0]['dataset']
examples = data['datasets'][0]['examples']

print(f"Dataset: {dataset_name}")
print(f"Number of examples: {len(examples)}")

## Data Structure Exploration

Each example in the CLEAR Corpus has:
- `input`: The text excerpt
- `output`: Human readability score (1-100 scale, higher = more readable)
- `metadata_text_id`: Unique identifier
- `metadata_rater_agreement`: Inverse of standard error (higher = more agreement)
- `metadata_num_sentences`: Number of sentences
- `metadata_num_words`: Number of words
- `metadata_lexile_band`: Lexile reading level band
- `metadata_domain`: Text domain (e.g., 'Lit' for Literature)
- `metadata_pub_year`: Publication year
- `metadata_bt_easiness_original`: Original BT_easiness score from Rasch model

In [ ]:
# Explore the structure of the first example
print("=== First Example Structure ===")
first_example = examples[0]
for key in first_example.keys():
    value = first_example[key]
    if key == 'input':
        # Truncate long text
        print(f"{key}: {value[:100]}...")
    else:
        print(f"{key}: {value}")

In [ ]:
# Display all examples in a readable format
print("=== All Examples ===")
for i, ex in enumerate(examples):
    print(f"\n--- Example {i+1} ---")
    print(f"Text ID: {ex['metadata_text_id']}")
    print(f"Readability Score: {ex['output']}")
    print(f"Domain: {ex['metadata_domain']}")
    print(f"Lexile Band: {ex['metadata_lexile_band']}")
    print(f"Sentences: {ex['metadata_num_sentences']}, Words: {ex['metadata_num_words']}")
    print(f"Rater Agreement: {ex['metadata_rater_agreement']:.3f}")
    print(f"Text preview: {ex['input'][:150]}...")

## Data Analysis and Visualization

Now let's analyze the readability scores and visualize the data.

In [ ]:
# Extract scores and compute statistics
scores = [float(ex['output']) for ex in examples]
num_sentences = [ex['metadata_num_sentences'] for ex in examples]
num_words = [ex['metadata_num_words'] for ex in examples]
rater_agreements = [ex['metadata_rater_agreement'] for ex in examples]

print("=== Dataset Statistics (Mini) ===")
print(f"Readability Scores:")
print(f"  Mean: {np.mean(scores):.2f}")
print(f"  Std:  {np.std(scores):.2f}")
print(f"  Min:  {np.min(scores):.2f}")
print(f"  Max:  {np.max(scores):.2f}")
print(f"\nText Statistics:")
print(f"  Avg Sentences: {np.mean(num_sentences):.1f}")
print(f"  Avg Words: {np.mean(num_words):.1f}")
print(f"\nRater Agreement:")
print(f"  Mean: {np.mean(rater_agreements):.3f}")

In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Readability Scores
axes[0].bar(range(len(scores)), scores, color='skyblue', edgecolor='black')
axes[0].axhline(y=np.mean(scores), color='red', linestyle='--', label=f'Mean: {np.mean(scores):.1f}')
axes[0].set_xlabel('Example Index')
axes[0].set_ylabel('Readability Score (1-100)')
axes[0].set_title('Readability Scores for Mini Dataset')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 2. Text Length Distribution (Words)
axes[1].bar(range(len(num_words)), num_words, color='lightgreen', edgecolor='black')
axes[1].axhline(y=np.mean(num_words), color='red', linestyle='--', label=f'Mean: {np.mean(num_words):.0f}')
axes[1].set_xlabel('Example Index')
axes[1].set_ylabel('Number of Words')
axes[1].set_title('Text Length (Word Count)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# 3. Rater Agreement
axes[2].bar(range(len(rater_agreements)), rater_agreements, color='coral', edgecolor='black')
axes[2].axhline(y=np.mean(rater_agreements), color='blue', linestyle='--', label=f'Mean: {np.mean(rater_agreements):.3f}')
axes[2].set_xlabel('Example Index')
axes[2].set_ylabel('Rater Agreement')
axes[2].set_title('Rater Agreement (Higher = Better)')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Additional analysis: Relationship between text length and readability
print("=== Relationship Analysis ===")
print("\nText Length vs Readability Score:")
for i, ex in enumerate(examples):
    score = float(ex['output'])
    words = ex['metadata_num_words']
    sentences = ex['metadata_num_sentences']
    avg_sentence_len = words / sentences if sentences > 0 else 0
    print(f"  Example {i+1}: Score={score:.1f}, Words={words}, Avg Sent/Len={avg_sentence_len:.1f}")

In [ ]:
# Create a summary table
print("=== Summary Table ===")
print(f"{'ID':<6} {'Score':<8} {'Words':<7} {'Sents':<7} {'Lexile':<8} {'Domain':<6} {'Year':<6}")
print("-" * 60)
for ex in examples:
    print(f"{ex['metadata_text_id']:<6} {float(ex['output']):<8.1f} {ex['metadata_num_words']:<7} {ex['metadata_num_sentences']:<7} {ex['metadata_lexile_band']:<8} {ex['metadata_domain']:<6} {ex['metadata_pub_year']:<6}")

## Conclusion

This demo notebook has shown:

1. **Data Loading**: Successfully loaded the CLEAR Corpus mini dataset from GitHub URL with local fallback
2. **Data Structure**: Each example contains the text, human readability score, and rich metadata
3. **Basic Statistics**: Computed mean, std, min, max for readability scores
4. **Visualization**: Created bar charts showing scores, text lengths, and rater agreement
5. **Analysis**: Explored relationships between text characteristics and readability

The full dataset contains **4,724 examples** with diverse texts spanning 250+ years. This mini dataset (3 examples) demonstrates the structure and can be used for testing pipelines before running on the full dataset.